# MLflow Serving Walkthrough

Полный процесс запуска обученной модели через ml-server:

| Секция | Этап | Описание |
|--------|------|----------|
| 0 | — | Конфигурация |
| 1 | STAGE 0 | Проверка инфраструктуры (ml-server, MLflow, Redis) |
| 2 | Pre-STAGE 3 | MLflow Registry: модели, версии, bundle config |
| 3 | STAGE 1–2 | POST /predict → task_id (HTTP 202) |
| 4 | STAGE 7 | Polling до результата |
| 5 | STAGE 5–6 | Визуализация прогноза и метрики качества |

**Требования к окружению:**
- Docker Compose запущен (`docker compose up -d` в `ml-server/`)
- MLflow UI доступен на `http://localhost:5050`
- ml-server доступен на `http://localhost:8030`
- Модель зарегистрирована в MLflow Registry с alias `Production`

---
## Секция 0 — Конфигурация

In [1]:
import os
import json
import time
import tempfile
import textwrap
from datetime import datetime, timezone

import requests
import pandas as pd
import plotly.graph_objects as go
import mlflow
from mlflow.tracking import MlflowClient

# ── Endpoints ────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
ML_SERVER_BASE_URL  = os.getenv("ML_SERVER_BASE_URL",  "http://localhost:8030")
REDIS_HOST          = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT          = int(os.getenv("REDIS_PORT", "6379"))

# ── Predict parameters ───────────────────────────────────────────────────────
MODEL_ID = os.getenv(
    "ML_SERVER_MODEL_ID",
    "root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt",
)
OBJECT_REFERENCE = os.getenv(
    "ML_SERVER_OBJECT_REFERENCE",
    "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
)
VERSION_ALIAS    = os.getenv("ML_SERVER_VERSION_ALIAS", "Production")
POLL_INTERVAL    = float(os.getenv("ML_SERVER_POLL_INTERVAL", "2"))
MAX_ATTEMPTS     = int(os.getenv("ML_SERVER_MAX_ATTEMPTS", "60"))

# ── Derived URLs ─────────────────────────────────────────────────────────────
PREDICT_URL       = f"{ML_SERVER_BASE_URL}/predict"
RUNTIME_STATUS_URL = f"{ML_SERVER_BASE_URL}/ui/runtime-status"
MODELS_URL        = f"{ML_SERVER_BASE_URL}/ui/models"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

print("Configuration:")
print(f"  MLFLOW_TRACKING_URI : {MLFLOW_TRACKING_URI}")
print(f"  ML_SERVER_BASE_URL  : {ML_SERVER_BASE_URL}")
print(f"  MODEL_ID            : {MODEL_ID}")
print(f"  VERSION_ALIAS       : {VERSION_ALIAS}")

Configuration:
  MLFLOW_TRACKING_URI : http://localhost:5050
  ML_SERVER_BASE_URL  : http://localhost:8030
  MODEL_ID            : root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt
  VERSION_ALIAS       : Production


---
## Секция 1 — Проверка инфраструктуры (STAGE 0)

Проверяем, что все три сервиса доступны перед запуском пайплайна.

In [2]:
def _check(label: str, fn):
    try:
        ok, detail = fn()
        status = "✅" if ok else "❌"
        print(f"{status}  {label}: {detail}")
        return ok
    except Exception as exc:
        print(f"❌  {label}: {type(exc).__name__}: {exc}")
        return False


def check_mlserver():
    r = requests.get(RUNTIME_STATUS_URL, timeout=5)
    data = r.json() if r.headers.get("content-type", "").startswith("application/json") else {}
    return r.status_code == 200, f"HTTP {r.status_code}  workers={data.get('workers', '?')}  uptime={data.get('uptime', '?')}"


def check_mlflow():
    r = requests.get(f"{MLFLOW_TRACKING_URI}/health", timeout=5)
    return r.status_code == 200, f"HTTP {r.status_code}  body={r.text.strip()[:60]}"


def check_redis():
    import redis as redis_lib
    r = redis_lib.Redis(host=REDIS_HOST, port=REDIS_PORT, socket_connect_timeout=3)
    pong = r.ping()
    return pong, f"PING → {'PONG' if pong else 'no response'}  ({REDIS_HOST}:{REDIS_PORT})"


print("Infrastructure health check:")
print("-" * 55)
ok_server = _check("ml-server          ", check_mlserver)
ok_mlflow = _check("MLflow Registry    ", check_mlflow)
ok_redis  = _check("Redis              ", check_redis)
print("-" * 55)

all_ok = ok_server and ok_mlflow and ok_redis
if all_ok:
    print("\nAll services are UP — ready to proceed.")
else:
    print("\n⚠  One or more services are DOWN.")
    print("   Run: cd ml-server && docker compose up -d")

Infrastructure health check:
-------------------------------------------------------
✅  ml-server          : HTTP 200  workers=?  uptime=?
✅  MLflow Registry    : HTTP 200  body=OK
✅  Redis              : PING → PONG  (localhost:6379)
-------------------------------------------------------

All services are UP — ready to proceed.


---
## Секция 2 — MLflow Registry: модели, версии, bundle config (pre-STAGE 3)

Просматриваем зарегистрированные модели и загружаем `cache_config.json` для выбранного alias.

In [3]:
# ── List all registered models ────────────────────────────────────────────────
registered = client.search_registered_models()

rows = []
for rm in registered:
    aliases_flat = ", ".join(
        f"{alias}=v{ver}" for ver, aliases in (rm.aliases or {}).items() for alias in aliases
    ) if rm.aliases else "—"
    rows.append({
        "model_id":      rm.name,
        "latest_version": rm.latest_versions[-1].version if rm.latest_versions else "—",
        "aliases":       aliases_flat,
        "created":       datetime.fromtimestamp(rm.creation_timestamp / 1000, tz=timezone.utc).strftime("%Y-%m-%d") if rm.creation_timestamp else "—",
    })

df_models = pd.DataFrame(rows)
print(f"Registered models in MLflow Registry ({MLFLOW_TRACKING_URI}):")
if df_models.empty:
    print("  No registered models found.")
else:
    display(df_models)

Registered models in MLflow Registry (http://localhost:5050):


,model_id,latest_version,aliases,created
0,root_FP_PROJECT_AKMOLA_VostVet_VES_models_P_wa...,3,3=vProduction,2026-05-05
1,кек,—,—,2026-05-07


In [4]:
# ── Versions for selected MODEL_ID ────────────────────────────────────────────
try:
    versions = client.search_model_versions(f"name='{MODEL_ID}'")
    rows_v = []
    for v in versions:
        rows_v.append({
            "version":    v.version,
            "run_id":     v.run_id,
            "status":     v.status,
            "aliases":    ", ".join(v.aliases) if v.aliases else "—",
            "created":    datetime.fromtimestamp(v.creation_timestamp / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M") if v.creation_timestamp else "—",
        })
    df_versions = pd.DataFrame(rows_v).sort_values("version", ascending=False)
    print(f"Versions for model '{MODEL_ID}':")
    display(df_versions)

    # Resolve alias → version
    mv = client.get_model_version_by_alias(MODEL_ID, VERSION_ALIAS)
    ACTIVE_RUN_ID      = mv.run_id
    ACTIVE_VERSION     = mv.version
    print(f"\nAlias '{VERSION_ALIAS}' → version {ACTIVE_VERSION}  (run_id={ACTIVE_RUN_ID})")
except Exception as exc:
    print(f"⚠  Could not fetch versions for '{MODEL_ID}': {exc}")
    ACTIVE_RUN_ID  = None
    ACTIVE_VERSION = None

⚠  Could not fetch versions for 'root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt': 'version'


In [5]:
# ── Download and display cache_config.json from bundle artifact ───────────────
BUNDLE_CONFIG = None

if ACTIVE_RUN_ID:
    with tempfile.TemporaryDirectory() as tmpdir:
        try:
            local_path = mlflow.artifacts.download_artifacts(
                run_id=ACTIVE_RUN_ID,
                artifact_path="bundle/configuration/cache_config.json",
                dst_path=tmpdir,
            )
            with open(local_path) as f:
                BUNDLE_CONFIG = json.load(f)
        except Exception:
            # Some bundles use a nested bundle/bundle/ layout
            try:
                local_path = mlflow.artifacts.download_artifacts(
                    run_id=ACTIVE_RUN_ID,
                    artifact_path="bundle/bundle/configuration/cache_config.json",
                    dst_path=tmpdir,
                )
                with open(local_path) as f:
                    BUNDLE_CONFIG = json.load(f)
            except Exception as exc:
                print(f"⚠  Could not download cache_config.json: {exc}")

if BUNDLE_CONFIG:
    print("cache_config.json (raw):")
    print(json.dumps(BUNDLE_CONFIG, indent=2, ensure_ascii=False))

    # ── Summary table ─────────────────────────────────────────────────────────
    src = BUNDLE_CONFIG.get("sources", BUNDLE_CONFIG)
    summary = {
        "model_type":    [BUNDLE_CONFIG.get("model_type", "—")],
        "step (sec)":    [src.get("step", "—")],
        "input_range":   [src.get("input_range", "—")],
        "output_range":  [src.get("output_range", "—")],
        "fallback":      [BUNDLE_CONFIG.get("fallback", "—")],
        "weather":       ["yes" if src.get("weather") or BUNDLE_CONFIG.get("weather_lat") else "no"],
        "cmms":          ["yes" if src.get("cmms") or BUNDLE_CONFIG.get("cmms_url") else "no"],
    }
    print("\nKey parameters:")
    display(pd.DataFrame(summary))
else:
    print("No bundle config available.")

No bundle config available.


---
## Секция 3 — POST /predict: запуск задачи (STAGE 1–2)

Отправляем запрос на прогноз. worker получит задачу через TaskIQ, выполнит `sync_with_registry`, загрузит данные и запустит инференс.

In [6]:
start_payload = {
    "model_id":        MODEL_ID,
    "object_reference": OBJECT_REFERENCE,
    "model_selection": {"version_alias": VERSION_ALIAS},
}

print("POST", PREDICT_URL)
print("Payload:")
print(json.dumps(start_payload, indent=2, ensure_ascii=False))
print()

start_resp = requests.post(PREDICT_URL, json=start_payload, timeout=30)
start_data = start_resp.json() if start_resp.content else {}

print(f"HTTP {start_resp.status_code}")
print(json.dumps(start_data, indent=2, ensure_ascii=False))

if start_resp.status_code != 202:
    raise RuntimeError(f"Expected HTTP 202, got {start_resp.status_code}")

TASK_ID = start_data.get("task_id")
if not TASK_ID:
    raise RuntimeError(f"task_id missing in response: {start_data}")

print(f"\ntask_id = {TASK_ID}")

POST http://localhost:8030/predict
Payload:
{
  "model_id": "root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt",
  "object_reference": "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
  "model_selection": {
    "version_alias": "Production"
  }
}

HTTP 202
{
  "status": 202,
  "task_id": "f64471987d034935adc95a28d4f159c7",
  "object_reference": "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
  "state": "start"
}

task_id = f64471987d034935adc95a28d4f159c7


---
## Секция 4 — Polling до результата (STAGE 7)

Опрашиваем `/predict` с `task_id`. Ответы:
- `HTTP 202` — задача ещё выполняется
- `HTTP 200` — результат готов
- `HTTP 422/503/500` — ошибка

In [7]:
RESULT = None
FINAL_STATUS = None

print(f"Polling (max {MAX_ATTEMPTS} attempts, interval={POLL_INTERVAL}s):")
print("-" * 55)

for attempt in range(1, MAX_ATTEMPTS + 1):
    poll_resp = requests.post(PREDICT_URL, json={"task_id": TASK_ID}, timeout=30)
    poll_data = poll_resp.json() if poll_resp.content else {}
    FINAL_STATUS = poll_resp.status_code
    state = poll_data.get("state", "?")

    if poll_resp.status_code == 202:
        print(f"  [{attempt:>3}/{MAX_ATTEMPTS}] state={state} — still processing…")
        time.sleep(POLL_INTERVAL)
        continue

    # Terminal response
    print(f"  [{attempt:>3}/{MAX_ATTEMPTS}] HTTP {poll_resp.status_code}  state={state}")
    RESULT = poll_data

    if poll_resp.status_code == 503:
        print("\n⚠  HTTP 503 — model not found in MLflow Registry for the selected alias.")
        try:
            snap = requests.get(MODELS_URL, timeout=10).json()
            print("  /ui/models snapshot:")
            print(json.dumps(snap, indent=2, ensure_ascii=False))
        except Exception as e:
            print(f"  Could not fetch /ui/models: {e}")
    break
else:
    raise TimeoutError(f"Task did not complete after {MAX_ATTEMPTS} attempts")

print("-" * 55)
print(f"Final HTTP status: {FINAL_STATUS}")

Polling (max 60 attempts, interval=2.0s):
-------------------------------------------------------
  [  1/60] HTTP 503  state=done

⚠  HTTP 503 — model not found in MLflow Registry for the selected alias.
  /ui/models snapshot:
{
  "status": 200,
  "models": [],
  "total": 0,
  "updated_at": "2026-05-07T19:54:52.733796+00:00"
}
-------------------------------------------------------
Final HTTP status: 503


---
## Секция 5 — Визуализация прогноза и метрики качества (STAGE 5–6 outputs)

In [8]:
if RESULT is None or FINAL_STATUS != 200:
    print(f"⚠  No result to visualize (HTTP {FINAL_STATUS}).")
    if RESULT:
        print(json.dumps(RESULT, indent=2, ensure_ascii=False))
else:
    output_points = RESULT.get("output", [])
    if not output_points:
        print("⚠  'output' is empty in the result.")
    else:
        # output format: [[timestamp_ms, value, qds], ...]
        df_out = pd.DataFrame(output_points, columns=["ts_ms", "value", "qds"])
        df_out["ts"]     = pd.to_datetime(df_out["ts_ms"], unit="ms", utc=True)
        df_out["ts_local"] = df_out["ts"].dt.tz_convert(None)   # remove tz for plotly

        # QDS colour map
        QDS_COLOR = {0: "#2ecc71", 64: "#f39c12", 128: "#e74c3c", 256: "#9b59b6"}
        QDS_LABEL = {0: "BASE (OK)", 64: "NOT_TOPICAL", 128: "INVALID", 256: "MISSING"}

        fig = go.Figure()

        # Main forecast line
        fig.add_trace(go.Scatter(
            x=df_out["ts_local"],
            y=df_out["value"],
            mode="lines",
            name="Forecast",
            line=dict(color="#3498db", width=2),
        ))

        # Scatter markers coloured by QDS
        for qds_val, grp in df_out.groupby("qds"):
            color = QDS_COLOR.get(int(qds_val), "#bdc3c7")
            label = QDS_LABEL.get(int(qds_val), f"QDS={qds_val}")
            fig.add_trace(go.Scatter(
                x=grp["ts_local"],
                y=grp["value"],
                mode="markers",
                name=label,
                marker=dict(color=color, size=6, symbol="circle"),
            ))

        step_sec = None
        if BUNDLE_CONFIG:
            src = BUNDLE_CONFIG.get("sources", BUNDLE_CONFIG)
            step_sec = src.get("step")
        step_label = f"{step_sec // 3600}h" if step_sec and step_sec >= 3600 else (str(step_sec) + "s" if step_sec else "")

        fig.update_layout(
            title=dict(
                text=f"Forecast  |  model_id={MODEL_ID}  |  alias={VERSION_ALIAS}  |  step={step_label}",
                font=dict(size=14),
            ),
            xaxis_title="Time (UTC)",
            yaxis_title="Value",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            hovermode="x unified",
            template="plotly_white",
            height=450,
        )
        fig.show()
        print(f"Points: {len(df_out)}  |  time range: {df_out['ts_local'].min()} → {df_out['ts_local'].max()}")

⚠  No result to visualize (HTTP 503).
{
  "status": 503,
  "message": "MLFLOW is not available, so it is impossible to take values ​​at this time. Error: No cached MLflow bundle available for model_id=root_FP_PROJECT_AKMOLA_regions_North_Kazakhstan_load_models_P_watt selector=Production",
  "object_reference": "/root/FP/PROJECT/AKMOLA/@regions/KOKSHETAU/Load/P_load/archives/out_value",
  "task_id": "f64471987d034935adc95a28d4f159c7",
  "state": "done"
}


In [9]:
if RESULT and FINAL_STATUS == 200:
    # ── Top-level quality metrics ─────────────────────────────────────────────
    metrics = {
        "quality":                    [RESULT.get("quality")],
        "model_confidence":           [RESULT.get("model_confidence")],
        "planned_adjustments_applied":[RESULT.get("planned_adjustments_applied", 0)],
        "message":                    [RESULT.get("message") or "—"],
    }
    print("Quality metrics:")
    display(pd.DataFrame(metrics))

    # ── Input statistics ──────────────────────────────────────────────────────
    in_stats = RESULT.get("input_statistics")
    if in_stats:
        print("\nInput statistics:")
        display(pd.DataFrame([in_stats]))

    # ── Output statistics ─────────────────────────────────────────────────────
    out_stats = RESULT.get("output_statistics")
    if out_stats:
        print("\nOutput statistics:")
        display(pd.DataFrame([out_stats]))

    # ── Full raw result (collapsed) ───────────────────────────────────────────
    result_no_output = {k: v for k, v in RESULT.items() if k != "output"}
    print("\nFull result (output array omitted):")
    print(json.dumps(result_no_output, indent=2, ensure_ascii=False))